In [1]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import pyautogui
import math
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

devices = AudioUtilities.GetSpeakers()
volume = devices.EndpointVolume


d:\Data Science\OpenCV\venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
class load_model:
    def __init__(self):
        self.base_options = python.BaseOptions(
        model_asset_path="hand_landmarker.task"
        )
        self.options = vision.HandLandmarkerOptions(
        base_options=self.base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_hands=1,
        min_hand_detection_confidence=0.7,
        min_tracking_confidence=0.7
        )
    def get_model(self):
        return vision.HandLandmarker.create_from_options(self.options)

In [3]:
class get_data:
    def __init__(self,landmark,h,w):
        self.landmark=landmark
        self.h=h
        self.w=w
    def get_landmark(self,a):
        return [int(self.landmark[a].x*self.w),int(self.landmark[a].y*self.h),self.landmark[a].z]
    
    

In [4]:
class calculate:
    def distance(self,a, b):
        return math.hypot(a[0] - b[0], a[1] - b[1])
    def finger_open(self,tip, pip, wrist,hand_size):
        return self.distance(tip, wrist) > self.distance(pip, wrist) +  0.15 * hand_size
    def finger_open_angle(self,a,b,c):
        a=np.array(a)
        b=np.array(b)
        c=np.array(c)
        radians=np.arctan2(c[1]-b[1],c[0]-b[0])-np.arctan2(a[1]-b[1],a[0]-b[0])
        angle=np.abs(180*radians/np.pi)
        if angle>180:
            angle=360-angle
        return angle > 150  
    def finger_open_z(self,tip, pip):
        return tip[2] < pip[2] - 0.01


In [5]:
HAND_CONNECTIONS = [
    # Thumb
    (0, 1), (1, 2), (2, 3), (3, 4),

    # Index finger
    (0, 5), (5, 6), (6, 7), (7, 8),

    # Middle finger
    (0, 9), (9, 10), (10, 11), (11, 12),

    # Ring finger
    (0, 13), (13, 14), (14, 15), (15, 16),

    # Pinky
    (0, 17), (17, 18), (18, 19), (19, 20),

    # Palm connections
    (5, 9), (9, 13), (13, 17)
]

In [ ]:
class start:
    def __init__(self,model,prev_dist,last_media_action,cooldown,prevx,prevy,alpha,pinch_start,width,height,volume):
        self.prev_dist=prev_dist
        self.last_media_action=last_media_action
        self.cooldown=cooldown
        self.prevx=prevx
        self.prevy=prevy
        self.alpha=alpha
        self.pinch_start=pinch_start
        self.screen_width=width
        self.screen_height=height
        self.volume=volume
        self.model=model
    def start_capture(self):
        dis=calculate()
        start_time=time.time()
        cap=cv2.VideoCapture(0)
        self.volume.SetMute(0,None)
        while cap.isOpened():
            ret,frame=cap.read()
            if not ret:
                print("failed to get the frame")
                break
            frame = cv2.flip(frame, 1)  
            image=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            mp_image=mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=image
            )
            frame_timestamp_ms = int((time.time() - start_time) * 1000)
            result = self.model.detect_for_video(mp_image, frame_timestamp_ms)
            if result.hand_landmarks:
                landmark=result.hand_landmarks[0]
                # print(landmark)
                h,w,_=frame.shape
                for lm in landmark:
                    x,y=int(lm.x*w),int(lm.y*h)
                    cv2.circle(frame,(x,y),5,(0,255,0),-1)
                for s,e in HAND_CONNECTIONS:
                    sx,sy=int(landmark[s].x*w),int(landmark[s].y*h)
                    ex,ey=int(landmark[e].x*w),int(landmark[e].y*h)
                    cv2.line(frame,(sx,sy),(ex,ey),(255,0,0),2)
                lndmrk=get_data(landmark,h,w)
                index_mcp=lndmrk.get_landmark(5)
                pinky_pip=lndmrk.get_landmark(18)
                pinky_tip=lndmrk.get_landmark(20)
                pinky_mcp=lndmrk.get_landmark(17)
                index_pip=lndmrk.get_landmark(6)
                index_tip=lndmrk.get_landmark(8)
                mid_pip=lndmrk.get_landmark(10)
                mid_tip=lndmrk.get_landmark(12)
                ring_pip=lndmrk.get_landmark(14)
                ring_tip=lndmrk.get_landmark(16)
                ring_mcp=lndmrk.get_landmark(13)
                thumb_tip=lndmrk.get_landmark(4)
                thumb_ip=lndmrk.get_landmark(3)
                wrist=lndmrk.get_landmark(0)
                middle_mcp=lndmrk.get_landmark(9)
                hand_size = dis.distance(wrist, middle_mcp)  
               
                index_open = dis.finger_open(index_tip, index_pip, wrist,hand_size)
                mid_open   =dis.finger_open(mid_tip, mid_pip, wrist,hand_size)
                ring_open  = dis.finger_open_angle(ring_mcp,ring_pip, ring_tip) and dis.finger_open_z(ring_tip,ring_pip)
                pinky_open = dis.finger_open_angle(pinky_mcp,pinky_pip, pinky_tip) and dis.finger_open_z(pinky_tip,pinky_pip)
                thumb_open = dis.finger_open(thumb_tip, thumb_ip, wrist,hand_size)

                print(f"index_open:{index_open}")
                print(f"mid_open:{mid_open}")
                print(f"ring_open:{ring_open}")
                print(f"pinky_open:{pinky_open}")
                print(f"thumb_open:{thumb_open}")
             
                
                volume_mode =  index_open and thumb_open and not mid_open and not ring_open and not pinky_open
                play_pause_gesture = index_open and mid_open and ring_open and not pinky_open and not thumb_open
                next_track_gesture = index_open and mid_open and ring_open and pinky_open and not thumb_open
                prev_track_gesture = index_open and mid_open and ring_open and pinky_open and thumb_open
                mouse_mode=index_open and thumb_open and mid_open and not ring_open and not pinky_open
                print(f"volume_mode:{volume_mode}")
                print(f"play_pause_gesture: {play_pause_gesture}")
                print(f"next_track_gesture: {next_track_gesture}")
                print(f"prev_track_gesture: {prev_track_gesture}")
                print(f"mouse_mode: {mouse_mode}")
                print("\n")
                cv2.rectangle(frame,(int((w/2)-25),5),(int((w/2)-25),30),(255,255,255),-1)
                current_time = time.time()
                if volume_mode:
                    dist = dis.distance(index_tip,thumb_tip)
                    dist = min(dist*1.25, 220)
                    volume.SetMasterVolumeLevelScalar(dist / 220, None)
                    cv2.rectangle(frame,(20,int(430-dist)),(45,430),(0,0,0),-1)
                    cv2.rectangle(frame,(20,int(430-dist)),(45,430),(0,0,0),2)
                    cv2.putText(frame,f"Volume: {round((dist/220)*100,2)}%",(10,200),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))
                    cv2.putText(frame,f"Volume Mode",(int(w/2-15),20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))
                elif mouse_mode :
                    dist = dis.distance(index_mcp,thumb_tip)
                    cv2.circle(frame,(index_tip[0],index_tip[1]),5,(0,255,0),-1)
                    cv2.circle(frame,(thumb_tip[0],thumb_tip[1]),5,(0,255,0),-1)
                    cv2.circle(frame,(index_mcp[0],index_mcp[1]),5,(0,255,0),-1)
                    x = np.clip(index_tip[0], 0, w)
                    y = np.clip(index_tip[1], 0, h)
                    screen_x = np.interp(x, [0, w], [0, self.screen_width])
                    screen_y = np.interp(y, [0, h], [0, self.screen_height])
                    screen_x = self.prevx + self.alpha * (screen_x - self.prevx)
                    screen_y = self.prevy + self.alpha * (screen_y - self.prevy)
                    self.prevx, self.prevy = screen_x, screen_y
                    if dist<50:
                        if self.pinch_start is None:
                            self.pinch_start = time.time()
                    else :
                        if self.pinch_start:
                            duration = time.time() - self.pinch_start
                            if duration > 0.5:
                                pyautogui.doubleClick()
                            else:
                                pyautogui.click()
                            self.pinch_start = None
                    cv2.putText(frame,str(dist),(20,32),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,0))
                    pyautogui.moveTo(self.prevx, self.prevy, duration=0.01)
                    cv2.putText(frame,f"Mouse Mode",(int(w/2-15),20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))
                elif prev_track_gesture and current_time - self.last_media_action > self.cooldown:
                    pyautogui.press("prevtrack")
                    cv2.putText(frame,"Previous",(490,200),cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255))
                    self.last_media_action = current_time
                    cv2.putText(frame,f"Media Mode",(int(w/2-15),20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))

                elif next_track_gesture and current_time - self.last_media_action > self.cooldown:
                    pyautogui.press("nexttrack")
                    cv2.putText(frame,"Next",(490,200),cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255))
                    self.last_media_action = current_time
                    cv2.putText(frame,f"Media Mode",(int(w/2-15),20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))
                    
                elif play_pause_gesture and current_time - self.last_media_action > self.cooldown:
                    pyautogui.press("playpause")
                    self.last_media_action = current_time
                    cv2.putText(frame,f"Media Mode",(int(w/2-15),20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0))
                
            cv2.imshow("Frame",frame)
            if cv2.waitKey(10) & 0xFF==ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()

            

In [7]:
load=load_model()
model=load.get_model()

In [8]:
screen_width, screen_height = pyautogui.size()
stream=start(model,0,0,2,0,0,0.3,None,screen_width,screen_height,volume)

In [9]:
stream.start_capture()

index_open:True
mid_open:True
ring_open:True
pinky_open:True
thumb_open:True
volume_mode:False
play_pause_gesture: False
next_track_gesture: False
prev_track_gesture: True
mouse_mode: False


index_open:True
mid_open:True
ring_open:True
pinky_open:True
thumb_open:False
volume_mode:False
play_pause_gesture: False
next_track_gesture: True
prev_track_gesture: False
mouse_mode: False


index_open:True
mid_open:True
ring_open:True
pinky_open:True
thumb_open:False
volume_mode:False
play_pause_gesture: False
next_track_gesture: True
prev_track_gesture: False
mouse_mode: False


index_open:True
mid_open:True
ring_open:True
pinky_open:True
thumb_open:False
volume_mode:False
play_pause_gesture: False
next_track_gesture: True
prev_track_gesture: False
mouse_mode: False


index_open:True
mid_open:True
ring_open:True
pinky_open:True
thumb_open:False
volume_mode:False
play_pause_gesture: False
next_track_gesture: True
prev_track_gesture: False
mouse_mode: False


index_open:True
mid_open:True
ring_o